# TürkResearcher — Stage 1: Embedder Fine-Tuning

Fine-tunes `paraphrase-multilingual-mpnet-base-v2` on **633K Turkish thesis** title↔abstract pairs with **subject-aware hard negatives**, producing `trakad-embed-v2`.

**Önce:** Runtime → Change runtime type → **T4 GPU** seç.

**Akış (~3-4 saat):**
1. Setup + GPU check
2. Workspace + HF auth
3. Pull `abstracts_filtered.parquet` from HF Hub
4. Build (anchor, positive, hard_negative) triplets
5. SmokeIle 1K eğitim (~5 dk)
6. Full 633K eğitim (~2-3 saat)
7. Smoke retrieval (mpnet vs trakad-embed-v2)
8. HF Hub'a push: `hakansabunis/trakad-embed-v2`

## 1. Setup

In [ ]:
!nvidia-smi
!pip install -q sentence-transformers>=3.0 datasets huggingface_hub pyarrow pandas tqdm

## 2. Workspace + HF auth

In [ ]:
import os
WORK_DIR = '/content/embed_train'
os.makedirs(WORK_DIR, exist_ok=True)
print('Working dir:', WORK_DIR)
!df -h /content | tail -1

In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('HF write token: ')
login(hf_token, add_to_git_credential=False)

## 3. Pull filtered parquet from HF Hub

Source: `hakansabunis/tr-academic-research-agent-index/abstracts_filtered.parquet` (~1.5 GB)

In [ ]:
from huggingface_hub import hf_hub_download

PARQUET_PATH = hf_hub_download(
    repo_id='hakansabunis/tr-academic-research-agent-index',
    filename='abstracts_filtered.parquet',
    repo_type='dataset',
    local_dir=WORK_DIR,
)
print('Parquet:', PARQUET_PATH)
import os
print(f'Size: {os.path.getsize(PARQUET_PATH)/1024/1024:.0f} MB')

## 4. Build (anchor, positive, hard_negative) triplets

For each thesis: anchor = `title_tr`, positive = `abstract_tr`,  
hard_negative = abstract from a different thesis in the same subject area.

In [ ]:
import pandas as pd, random
from collections import defaultdict

MIN_ABSTRACT_CHARS = 200
MIN_TITLE_CHARS = 5
SEED = 42

df = pd.read_parquet(PARQUET_PATH, columns=['tez_no', 'title_tr', 'abstract_tr', 'subject'])
print(f'Loaded {len(df):,} rows')

df['title_tr']    = df['title_tr'].fillna('').astype(str).str.strip()
df['abstract_tr'] = df['abstract_tr'].fillna('').astype(str).str.strip()
df['subject']     = df['subject'].fillna('_unknown_').astype(str)

mask = (df['title_tr'].str.len() >= MIN_TITLE_CHARS) & (df['abstract_tr'].str.len() >= MIN_ABSTRACT_CHARS)
df = df[mask].reset_index(drop=True)
print(f'After cleanup: {len(df):,} rows')

by_subject = defaultdict(list)
for i, s in enumerate(df['subject'].values):
    by_subject[s].append(i)

rng = random.Random(SEED)
all_idx = list(range(len(df)))

anchors, positives, negs = [], [], []
for i in range(len(df)):
    anchors.append(df.at[i, 'title_tr'])
    positives.append(df.at[i, 'abstract_tr'])
    cands = by_subject[df.at[i, 'subject']]
    if len(cands) > 1:
        for _ in range(5):
            j = rng.choice(cands)
            if j != i: break
    else:
        for _ in range(5):
            j = rng.choice(all_idx)
            if j != i: break
    negs.append(df.at[j, 'abstract_tr'])

print(f'Built {len(anchors):,} triplets')

## 5. Smoke training (1K, ~5 dk)

Verify pipeline end-to-end before committing 3 hours of T4 time.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import torch, gc, math

# A100 (40 GB) recipe:
#   - max_seq=256 (full thesis abstract fits)
#   - batch=128 -> 383 in-batch negatives per anchor (stronger contrastive signal)
#   - use_amp=True (fp16 mixed precision; A100 tensor cores fly)
BASE_MODEL  = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
MAX_SEQ     = 256
SMOKE_BATCH = 128

gc.collect(); torch.cuda.empty_cache()

smoke_n = 1000
smoke_examples = [
    InputExample(texts=[anchors[i], positives[i], negs[i]])
    for i in range(smoke_n)
]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} · max_seq={MAX_SEQ} · batch={SMOKE_BATCH} · amp=on')

smoke_model = SentenceTransformer(BASE_MODEL, device=device)
smoke_model.max_seq_length = MAX_SEQ

smoke_loader = DataLoader(smoke_examples, shuffle=True, batch_size=SMOKE_BATCH)
smoke_loss   = losses.MultipleNegativesRankingLoss(smoke_model)
smoke_steps  = math.ceil(smoke_n / SMOKE_BATCH)

smoke_model.fit(
    train_objectives=[(smoke_loader, smoke_loss)],
    epochs=1,
    warmup_steps=int(smoke_steps * 0.03),
    optimizer_params={'lr': 2e-5},
    use_amp=True,
    show_progress_bar=True,
    output_path=f'{WORK_DIR}/smoke_out',
)
print('Smoke OK — pipeline works end-to-end.')

del smoke_model, smoke_loader, smoke_loss, smoke_examples
gc.collect(); torch.cuda.empty_cache()

## 6. Full training (633K, 1 epoch, ~2-3 saat on T4)

Using the same recipe with the full triplet set. Resumable: re-running this cell after disconnect picks up by re-reading the parquet but **does not** continue mid-epoch (we accept the cost; one epoch is short enough).

In [ ]:
import gc
gc.collect(); torch.cuda.empty_cache()

# A100 40 GB recipe.
# batch=128 fits comfortably with max_seq=256 + amp. Larger batch = stronger
# in-batch negative signal for MultipleNegativesRankingLoss (this is the main
# quality lever we got upgrading from T4).
BATCH_SIZE = 128
EPOCHS     = 1
LR         = 2e-5
MAX_SEQ    = 256
OUT_DIR    = f'{WORK_DIR}/trakad-embed-v2'

examples = [
    InputExample(texts=[anchors[i], positives[i], negs[i]])
    for i in range(len(anchors))
]
print(f'Total examples: {len(examples):,}')

model = SentenceTransformer(BASE_MODEL, device=device)
model.max_seq_length = MAX_SEQ

loader  = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)
loss_fn = losses.MultipleNegativesRankingLoss(model)

steps_per_epoch = math.ceil(len(examples) / BATCH_SIZE)
warmup_steps    = int(steps_per_epoch * EPOCHS * 0.03)
print(f'Steps/epoch: {steps_per_epoch:,}  · warmup: {warmup_steps}  · amp=on')

import time; t0 = time.time()
model.fit(
    train_objectives=[(loader, loss_fn)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': LR},
    use_amp=True,
    output_path=OUT_DIR,
    show_progress_bar=True,
)
print(f'Done in {(time.time()-t0)/60:.1f} min — saved to {OUT_DIR}')

## 7. Smoke retrieval — baseline mpnet vs trakad-embed-v2

In [ ]:
queries = [
    'derin öğrenme ile sel tahmini',
    'Türkçe doğal dil işleme metodları',
    'kalp damar hastalıkları teşhis yöntemleri',
    'yenilenebilir enerji rüzgar türbin',
    'üniversite öğrencilerinin akademik başarısı',
]

# Sample 5K abstracts as a fast eval corpus
sample_idx = rng.sample(all_idx, k=min(5000, len(all_idx)))
sample_titles    = [df.at[i, 'title_tr']    for i in sample_idx]
sample_abstracts = [df.at[i, 'abstract_tr'] for i in sample_idx]
sample_docs      = [f'{t}\n\n{a}' for t, a in zip(sample_titles, sample_abstracts)]

print('Encoding 5K corpus with v2 (trakad-embed)...')
v2 = SentenceTransformer(OUT_DIR, device=device)
v2_corpus = v2.encode(sample_docs, normalize_embeddings=True, batch_size=128, show_progress_bar=True)

print('Encoding 5K corpus with v1 (mpnet baseline)...')
v1 = SentenceTransformer(BASE_MODEL, device=device)
v1_corpus = v1.encode(sample_docs, normalize_embeddings=True, batch_size=128, show_progress_bar=True)

import numpy as np
for q in queries:
    qv2 = v2.encode(q, normalize_embeddings=True)
    qv1 = v1.encode(q, normalize_embeddings=True)
    s2 = (v2_corpus @ qv2)
    s1 = (v1_corpus @ qv1)
    top2 = np.argsort(-s2)[:3]
    top1 = np.argsort(-s1)[:3]
    print(f'\nQ: {q!r}')
    print('  v2 (trakad):')
    for j in top2:
        print(f'    [{s2[j]:.3f}] {sample_titles[j][:90]}')
    print('  v1 (mpnet):')
    for j in top1:
        print(f'    [{s1[j]:.3f}] {sample_titles[j][:90]}')

## 8. Push to HF Hub: `hakansabunis/trakad-embed-v2`

In [ ]:
from huggingface_hub import HfApi

REPO = 'hakansabunis/trakad-embed-v2'
api = HfApi(token=hf_token)

api.create_repo(REPO, repo_type='model', exist_ok=True, private=False, token=hf_token)
print(f'Uploading {OUT_DIR} -> {REPO} ...')
api.upload_folder(
    folder_path=OUT_DIR,
    repo_id=REPO,
    repo_type='model',
    commit_message='trakad-embed-v2 — Turkish academic SimCSE fine-tune (633K theses, MultipleNegativesRankingLoss + subject-aware hard negatives)',
    token=hf_token,
)
print(f'\n✓ Done. https://huggingface.co/{REPO}')